# Part 5 — Advanced psycopg 3 Features

This notebook covers features that are either new in psycopg 3 or that help you squeeze maximum performance out of PostgreSQL:

- pgvector embeddings storage and similarity search
- Streaming large result sets with server-side cursors
- Chunk-iterator pattern for memory-efficient processing
- Connection pool and auto-prepare tuning
- Pipeline mode (built-in for upsert loops)
- Efficient parameterized queries and JSONB filtering

**Prerequisites:** [Part 1](Part1_Getting_Started.ipynb)

In [ ]:
import pandas as pd
import numpy as np
from postgres_connector import PostgresConnector

pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    schema="tutorial",
)
pg.execute_query("CREATE SCHEMA IF NOT EXISTS tutorial;")

## 1. pgvector — Storing and Querying AI Embeddings

`PostgresConnector` automatically detects columns whose values are lists of floats and maps them to the PostgreSQL `VECTOR` type (provided `pgvector` is installed and the extension is enabled).

This enables native nearest-neighbour search directly in SQL.

### 1a. Enable the extension and store embeddings

In [ ]:
# Enable pgvector (requires the extension to be installed in PostgreSQL)
try:
    pg.execute_query("CREATE EXTENSION IF NOT EXISTS vector;")
    print("pgvector extension ready.")
except Exception as e:
    print(f"Could not enable pgvector: {e}")
    print("Install it with: apt install postgresql-16-pgvector (or equivalent)")

In [ ]:
# In practice embeddings come from a model like text-embedding-3-small (1536 dims).
# We use 4-dimensional vectors here to keep the example readable.
products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "name": ["Keyboard", "Mouse", "Monitor", "Headset", "Webcam"],
    "embedding": [
        [0.1, 0.9, 0.2, 0.5],
        [0.2, 0.8, 0.3, 0.4],
        [0.9, 0.1, 0.8, 0.2],
        [0.3, 0.7, 0.1, 0.6],
        [0.4, 0.6, 0.9, 0.1],
    ],
})

# The connector detects list-of-floats → creates a VECTOR(4) column
pg.upsert_data(products, "product_embeddings", primary_key="product_id")
print("Embeddings stored.")

### 1b. Create an HNSW index for fast approximate search

Without an index, every similarity query does a full table scan (exact but slow).  
An HNSW index makes nearest-neighbour searches sub-linear — essential at scale.

In [ ]:
try:
    pg.execute_query("""
        CREATE INDEX IF NOT EXISTS product_embedding_hnsw
        ON product_embeddings
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
    """)
    print("HNSW index created.")
except Exception as e:
    print(f"Index creation skipped: {e}")

### 1c. Nearest-neighbour similarity search

In [ ]:
# Find the 3 products most similar to a query vector
# Note: psycopg 3 / SQLAlchemy do not auto-cast plain lists to VECTOR,
# so cast explicitly in the query.
query_vector = "[0.15, 0.85, 0.25, 0.45]"

similar = pg.get_data(f"""
    SELECT
        product_id,
        name,
        1 - (embedding <=> '{query_vector}'::vector) AS cosine_similarity
    FROM product_embeddings
    ORDER BY embedding <=> '{query_vector}'::vector
    LIMIT 3;
""")
print(similar)

### 1d. Updating embeddings

Use `upsert_data` with `conflict_strategy='last'` — the VECTOR column updates just like any other column.

In [ ]:
# Re-embed product 1 after model retraining
updated_embedding = pd.DataFrame({
    "product_id": [1],
    "name":       ["Keyboard"],
    "embedding":  [[0.15, 0.88, 0.22, 0.51]],
})

pg.upsert_data(updated_embedding, "product_embeddings",
               primary_key="product_id", conflict_strategy="last")
print("Embedding updated.")

## 2. Streaming Large Result Sets

By default `get_data()` fetches all rows into memory before returning the DataFrame.  
For large tables this can exhaust RAM. Pass `stream=True` to use a **psycopg 3 server-side cursor** that fetches rows in small batches.

The returned DataFrame is identical — you only change memory behaviour, not the API.

In [ ]:
# Seed a large table
large = pd.DataFrame({
    "id":    range(500_000),
    "value": np.random.rand(500_000),
})
pg.replace_table(large, "large_table", primary_key="id")
print(f"Seeded {len(large):,} rows.")

In [ ]:
import tracemalloc

# Without streaming — loads all 500 000 rows into memory at once
tracemalloc.start()
df_normal = pg.get_data("SELECT * FROM large_table;")
_, peak_normal = tracemalloc.get_traced_memory()
tracemalloc.stop()

# With streaming — server-side cursor, memory stays bounded
tracemalloc.start()
df_streamed = pg.get_data("SELECT * FROM large_table;", stream=True, stream_buffer=10_000)
_, peak_streamed = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak memory — normal:   {peak_normal / 1e6:.1f} MB")
print(f"Peak memory — streamed: {peak_streamed / 1e6:.1f} MB")
assert len(df_normal) == len(df_streamed)

## 3. Chunk Iterator — Processing Without Loading Everything

`get_data_chunks()` yields one `pd.DataFrame` per chunk.  
Use it when you want to process or transform rows incrementally — writing to Parquet, running batch inference, building rolling aggregates — without ever holding the full result set in memory.

In [ ]:
# Example: compute a running sum across chunks without loading everything
running_total = 0.0
chunk_count = 0

for chunk in pg.get_data_chunks("SELECT value FROM large_table;", chunksize=50_000):
    running_total += chunk["value"].sum()
    chunk_count += 1

print(f"Processed {chunk_count} chunks")
print(f"Grand total: {running_total:.4f}")

In [ ]:
# Example: export to Parquet files — one file per chunk
import os
os.makedirs("/tmp/export", exist_ok=True)

for i, chunk in enumerate(pg.get_data_chunks("SELECT * FROM large_table;", chunksize=100_000)):
    path = f"/tmp/export/part_{i:04d}.parquet"
    chunk.to_parquet(path, index=False)
    print(f"  Written {len(chunk):,} rows → {path}")

print("Export complete.")

## 4. Connection Pool Tuning

The connector exposes the most important SQLAlchemy pool parameters directly in its constructor.  
Adjust them to match your workload.

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `pool_size` | 5 | Increase for high-concurrency services |
| `max_overflow` | 10 | Extra connections allowed during spikes |
| `pool_timeout` | 30.0 s | Reduce to fail-fast in latency-sensitive paths |
| `prepare_threshold` | 5 | Lower (e.g. 1) to prepare queries sooner; raise to disable |

`prepare_threshold` is a psycopg 3 feature: after a statement is executed this many times on the same connection, psycopg 3 sends it to PostgreSQL as a prepared statement. Subsequent executions skip the parse/plan phase entirely, cutting latency for repeated queries (like upsert chunk loops).

In [ ]:
# Production configuration example
pg_prod = PostgresConnector(
    host="db.example.com",
    database="prod",
    username="app_user",
    password="...",
    pool_size=20,           # up to 20 persistent connections
    max_overflow=30,        # burst headroom
    pool_timeout=5.0,       # fail fast under load
    prepare_threshold=1,    # prepare every statement on first use
)

## 5. Pipeline Mode — Automatic for Chunk Loops

When `upsert_data()` or `delete_and_insert()` iterate over multiple chunks, the connector enters **psycopg 3 pipeline mode** automatically.

In pipeline mode all INSERT / upsert statements in the loop are sent to the server in one batch, without waiting for an acknowledgement between each one. PostgreSQL processes them in parallel. This can reduce round-trip overhead by an order of magnitude for datasets that split into many chunks.

**You don't need to do anything.** It's on by default when you call `upsert_data()` with a large DataFrame.

In [ ]:
import time

# A DataFrame wide enough to force many chunks (pipeline becomes meaningful)
wide = pd.DataFrame(
    np.random.rand(10_000, 50),
    columns=[f"c{i}" for i in range(50)],
)
wide.insert(0, "id", range(10_000))

# _safe_chunk_size with 51 cols → 627 rows/chunk → ~16 chunks total
n_cols = len(wide.columns)
print(f"Columns: {n_cols}, chunk size: {PostgresConnector._safe_chunk_size(n_cols)} rows")

t0 = time.perf_counter()
pg.upsert_data(wide, "pipeline_demo", primary_key="id")
print(f"Upserted in {time.perf_counter() - t0:.2f}s  (pipeline active across all chunks)")

## 6. Advanced Parameterized Queries

Both `execute_query()` and `get_data()` use SQLAlchemy's `text()` binding, which means you get full support for all PostgreSQL parameter types — including arrays, which map cleanly to Python lists.

In [ ]:
# Array parameter → ANY(:ids) — safe against SQL injection
result = pg.get_data(
    "SELECT id, value FROM large_table WHERE id = ANY(:ids) ORDER BY id;",
    params={"ids": [1, 100, 999]},
)
print(result)

In [ ]:
# JSONB filter with parameterized cast — avoids f-string injection risk
# (assumes the 'users' table from Part 4 exists)
verified_users = pg.get_data(
    """
    SELECT user_id, profile->>'role' AS role
    FROM users
    WHERE (profile->>'verified')::boolean = :flag;
    """,
    params={"flag": True},
)
print(verified_users)

## 7. Creating Indexes After a Bulk Load

For large `replace_table()` loads it is faster to create indexes **after** the data is in place (PostgreSQL bulk-builds the index rather than maintaining it during insert).

In [ ]:
# 1. Load data first
pg.replace_table(large, "large_table", primary_key="id")

# 2. Then build indexes
pg.execute_query("""
    CREATE INDEX IF NOT EXISTS idx_large_value
    ON large_table (value);
""")

# 3. Verify the index is used
plan = pg.get_data("EXPLAIN SELECT * FROM large_table WHERE value < 0.01;")
print(plan["QUERY PLAN"].to_string(index=False))

## Cleanup

In [ ]:
for tbl in ["product_embeddings", "large_table", "pipeline_demo"]:
    pg.execute_query(f"DROP TABLE IF EXISTS {tbl};")

pg.dispose()
print("Done.")

## Summary

| Feature | How to use |
|---------|------------|
| pgvector storage | Pass `list[float]` column — mapped to `VECTOR(n)` automatically |
| Similarity search | Use `<=>` (cosine), `<->` (L2), `<#>` (inner product) operators |
| HNSW index | `CREATE INDEX … USING hnsw (col vector_cosine_ops)` via `execute_query` |
| Streaming fetch | `get_data(sql, stream=True, stream_buffer=N)` |
| Chunk iterator | `for chunk in get_data_chunks(sql, chunksize=N): …` |
| Pool tuning | `pool_size`, `max_overflow`, `pool_timeout` in constructor |
| Auto-prepare | `prepare_threshold=N` — lower = prepare sooner |
| Pipeline mode | Automatic inside `upsert_data` / `delete_and_insert` chunk loops |
| Array params | `WHERE col = ANY(:list_param)` with `params={"list_param": [...]}` |
| Post-load indexes | Call `execute_query(CREATE INDEX …)` after `replace_table()` |

---

**You've completed the full tutorial series.**

| # | Notebook | Topic |
|---|----------|-------|
| 1 | [Part1_Getting_Started](Part1_Getting_Started.ipynb) | Connection, execute_query, get_data |
| 2 | [Part2_Upsert_and_Schema](Part2_Upsert_and_Schema.ipynb) | Upsert strategies, schema evolution |
| 3 | [Part3_Bulk_Operations](Part3_Bulk_Operations.ipynb) | replace_table, delete_and_insert |
| 4 | [Part4_Edge_Cases](Part4_Edge_Cases.ipynb) | NaN, numpy, casing, empty frames … |
| 5 | Part5_Advanced_psycopg3 ← you are here | Streaming, pgvector, pipeline, pool |